# 03B 模型验证与调参：这个高分可信吗？

> 🟢 **Level A · 必须掌握** | 完成标准：解释 `train → CV/tune → untouched test`。


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline


## 三种数据各做什么？
训练折拟合模型和预处理参数；验证折选择超参数；留出的测试集最后评估一次。scikit-learn 使用负 MAE 是为了统一“越大越好”，报告误差时取负号还原。折间标准差反映这些划分的波动，不是置信区间。


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split,KFold,cross_validate,RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score
df=pd.read_csv('https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv')
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
X=df[features].copy(); y=df['CO2-1 bar (mol/kg)']
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
cv=KFold(5,shuffle=True,random_state=42)
m=RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)
m=make_pipeline(SimpleImputer(strategy='median'), m)
s=cross_validate(m,X_dev,y_dev,cv=cv,scoring={'r2':'r2','mae':'neg_mean_absolute_error'})
print('CV R² =',s['test_r2'].mean(),'+/-',s['test_r2'].std())
print('CV MAE =',(-s['test_mae']).mean(),'+/-',(-s['test_mae']).std())


In [ ]:
param={'n_estimators':[150,300,500],'max_depth':[None,6,10,16],'min_samples_leaf':[1,2,4],'max_features':['sqrt',0.7,1.0]}
param={'randomforestregressor__'+k:v for k,v in param.items()}
search=RandomizedSearchCV(make_pipeline(SimpleImputer(strategy='median'),RandomForestRegressor(random_state=42,n_jobs=1)),param,n_iter=12,cv=cv,scoring='neg_mean_absolute_error',random_state=42,n_jobs=-1).fit(X_dev,y_dev)
p=search.best_estimator_.predict(X_test)
print(search.best_params_)
print('test MAE =',mean_absolute_error(y_test,p),'test R² =',r2_score(y_test,p))


## 常见 leakage
near-duplicates 跨 split、family/topology 泄漏、split 前在全数据上 preprocessing/selection、target-derived feature、调参时反复看 test。


## family-aware split：把原则变成操作
随机划分回答同分布相似材料上的表现；分组划分回答未见家族上的表现。分组必须来自结构、linker 或 topology 元数据，不能把行号分段冒充化学家族。下面的人工组只演示 API，科研时替换为已核验的 COF family。


In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
toy_X = np.arange(48).reshape(24, 2)
toy_groups = np.repeat(['family_A','family_B','family_C','family_D','family_E','family_F'], 4)
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42).split(toy_X, groups=toy_groups))
assert set(toy_groups[tr]).isdisjoint(toy_groups[te])
for train_fold, val_fold in GroupKFold(3).split(toy_X[tr], groups=toy_groups[tr]):
    assert set(toy_groups[tr][train_fold]).isdisjoint(toy_groups[tr][val_fold])
print('train families:', sorted(set(toy_groups[tr])))
print('test families:', sorted(set(toy_groups[te])))


## 练习
先画出 train/CV/test 的职责，再运行搜索。说明随机划分与分组划分各自对应哪种泛化问题。记录划分索引、随机种子、预处理与参数空间；不要因为某个种子的测试分高就选择它。


## 数据来源与扩展阅读
[Dataset contracts / 数据使用约定](../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
